In [ ]:
# =====================================================================
# PHASE 1: LOAD & DIAGNOSE THE NYC AIRBNB DATA
# =====================================================================
import pandas as pd
import numpy as np

# Load the dataset (Make sure the filename matches what you uploaded)
df_airbnb = pd.read_csv('AB_NYC_2019.csv') # Change name if your file is named differently

print("📊 Dataset Shape:", df_airbnb.shape)
print("\n🔍 Missing Values Log:")
print(df_airbnb.isnull().sum())


📊 Dataset Shape: (14884, 16)

🔍 Missing Values Log:
id                                   0
name                                12
host_id                              0
host_name                           13
neighbourhood_group                  1
neighbourhood                        1
latitude                             1
longitude                            1
room_type                            1
price                                1
minimum_nights                       1
number_of_reviews                    1
last_review                       2456
reviews_per_month                 2456
calculated_host_listings_count       1
availability_365                     1
dtype: int64


In [ ]:
# =====================================================================
# PHASE 2: FIXING MISSING VALUES (IMPUTATION)
# =====================================================================
# 1. Fill missing text columns with placeholders
df_airbnb['name'].fillna('Unknown Listing', inplace=True)
df_airbnb['host_name'].fillna('Unknown Host', inplace=True)

# 2. If reviews_per_month is missing, it means 0 reviews have been made
df_airbnb['reviews_per_month'].fillna(0, inplace=True)

# 3. If last_review date is missing, set it to a placeholder date or string
df_airbnb['last_review'].fillna('No Reviews Recorded', inplace=True)

print("✅ Missing values handled! New missing count:")
print(df_airbnb.isnull().sum())


✅ Missing values handled! New missing count:
id                                0
name                              0
host_id                           0
host_name                         0
neighbourhood_group               1
neighbourhood                     1
latitude                          1
longitude                         1
room_type                         1
price                             1
minimum_nights                    1
number_of_reviews                 1
last_review                       0
reviews_per_month                 0
calculated_host_listings_count    1
availability_365                  1
dtype: int64


/tmp/ipykernel_7857/3552440517.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_airbnb['name'].fillna('Unknown Listing', inplace=True)
/tmp/ipykernel_7857/3552440517.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [ ]:
# =====================================================================
# PHASE 3: STANDARDIZATION & DEDUPLICATION
# =====================================================================
# Clean text columns: remove extra spaces and ensure standard casing
df_airbnb['neighbourhood_group'] = df_airbnb['neighbourhood_group'].str.strip().str.title()
df_airbnb['room_type'] = df_airbnb['room_type'].str.strip().str.capitalize()

# Check and remove duplicate rows
duplicates = df_airbnb.duplicated().sum()
print(f"Identified duplicate records: {duplicates}")
if duplicates > 0:
    df_airbnb.drop_duplicates(inplace=True)

print("✅ Text fields standardized and duplicates eliminated.")


Identified duplicate records: 0
✅ Text fields standardized and duplicates eliminated.


In [ ]:
# =====================================================================
# PHASE 4: OUTLIER CAPPING USING IQR METHOD
# =====================================================================
# Drop entries where price is 0 (physically impossible for commercial renting)
df_airbnb = df_airbnb[df_airbnb['price'] > 0]

# Calculate statistical upper bounds for realistic nightly pricing
Q1 = df_airbnb['price'].quantile(0.25)
Q3 = df_airbnb['price'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + (1.5 * IQR)
lowwer_limit = Q1 - (1.5 * IQR)

#print(f"Mathematical Lower limit for regular pricing: ${lowwer_limit:.2f}")

print(f"Mathematical Upper limit for regular pricing: ${upper_limit:.2f}")

# Keep only regular properties and filter out extreme data entry errors
df_cleaned_airbnb = df_airbnb[df_airbnb['price'] <= upper_limit].copy()

print(f"🏁 Original rows: {len(df_airbnb)} | Rows after removing price outliers: {len(df_cleaned_airbnb)}")

# Export clean file
df_cleaned_airbnb.to_csv('nyc_airbnb_cleaned.csv', index=False)
print("💾 Saved as 'nyc_airbnb_cleaned.csv'!")


Mathematical Upper limit for regular pricing: $327.50
🏁 Original rows: 14883 | Rows after removing price outliers: 14046
💾 Saved as 'nyc_airbnb_cleaned.csv'!


In [ ]:
## 🧹 Project 3 Conclusion - NYC Airbnb Cleaning Log

### ⚙️ Cleaning Methods Used:
# 1. **Structural Logic:** Filled missing values in `reviews_per_month` with `0`, treating properties with no booking history correctly.
# 2. **Data Consistency:** Standardized text strings across neighborhood columns to prevent duplicate regional groupings during future visualization steps.
# 3. **Outlier Filtering:** Applied Interquartile Range (IQR) logic to strip away invalid price records ($0 listings and extreme luxury outlier skewing anomalies).
